# Tema: Structured Streaming: estado y ventanas

## Objetivos
Ejecutar agregación incremental, observar métricas y trabajar con tiempo de evento.

## Conceptos importantes para el examen
Modos append/complete; watermark y datos tardíos; estado; el checkpoint pertenece a una consulta y no admite cualquier cambio de plan.

**Dificultad:** Intermedio · **Tiempo estimado:** 75 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_15_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
events = spark.createDataFrame([(i, i % 4, datetime(2026, 1, 1, 10, i), float(i * 10)) for i in range(1, 13)], "event_id INT, customer_id INT, event_time TIMESTAMP, amount DOUBLE")
events.write.format("delta").mode("overwrite").saveAsTable("events_source")

In [ ]:
# Requiere CREATE VOLUME en el schema; alternativa: usa un volumen autorizado.
spark.sql("CREATE VOLUME IF NOT EXISTS lab_files")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/lab_files"
dbutils.fs.mkdirs(BASE + "/landing")
CHECKPOINT = BASE + "/checkpoints/main"
print(BASE)

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Agregación acumulada
complete escribe el resultado completo de la agregación. Es apropiado aquí por el estado diminuto.

In [ ]:
def run_counts():
    q = (spark.readStream.table("events_source").groupBy("customer_id").count()
         .writeStream.format("delta").outputMode("complete")
         .option("checkpointLocation", CHECKPOINT).trigger(availableNow=True).toTable("customer_counts"))
    q.awaitTermination()
    return q
q = run_counts()
display(spark.table("customer_counts"))

### 2. Ventanas y watermark
append solo publica ventanas cerradas por el avance del watermark; no esperes todas las ventanas inmediatamente.

In [ ]:
windowed = (spark.readStream.table("events_source").withWatermark("event_time", "5 minutes")
            .groupBy(F.window("event_time", "5 minutes")).agg(F.sum("amount").alias("amount")))
qw = (windowed.writeStream.format("delta").outputMode("append")
      .option("checkpointLocation", BASE + "/checkpoints/windows").trigger(availableNow=True).toTable("window_totals"))
qw.awaitTermination()
display(spark.table("window_totals"))

### 3. Métricas

In [ ]:
print(q.lastProgress)
print(qw.lastProgress)
# Revisa numInputRows, durationMs, eventTime y stateOperators cuando existan.
# El último microbatch puede contener cero filas; mira también recentProgress.

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Añade tres eventos 13–15 y vuelve a ejecutar la agregación. La suma de count debe ser 15.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Compara el resultado incremental con un groupBy batch de toda la fuente.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Añade un evento de las 11:00 y ejecuta de nuevo la consulta de ventanas. Observa qué ventanas se publican.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Crea otra consulta que deduplique event_id dentro del watermark de 10 minutos; usa otro destino y checkpoint.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Inspecciona cada microbatch reciente y anota filas de entrada, duración y estado. Explica por qué no debes usar skipChangeCommits para propagar UPDATE.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** Reutiliza run_counts.

**Pista 2:** exceptAll en las dos direcciones.

**Pista 3:** El watermark depende del máximo tiempo observado, no del reloj del notebook.

**Pista 4:** dropDuplicatesWithinWatermark está disponible en DBR moderno; no garantiza deduplicación eterna.

**Pista 5:** Las métricas dependen del operador; skipChangeCommits omite commits de cambio.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
spark.createDataFrame([(i, i%4, datetime(2026,1,1,10,i), float(i*10)) for i in range(13,16)], events.schema).write.format("delta").mode("append").saveAsTable("events_source")
q = run_counts()
assert spark.table("customer_counts").agg(F.sum("count")).first()[0] == 15

### Solución 2

In [ ]:
expected = spark.table("events_source").groupBy("customer_id").count()
actual = spark.table("customer_counts")
assert expected.exceptAll(actual).count() == actual.exceptAll(expected).count() == 0

### Solución 3

In [ ]:
spark.createDataFrame([(99, 1, datetime(2026,1,1,11,0), 5.0)], events.schema).write.format("delta").mode("append").saveAsTable("events_source")
qw = (windowed.writeStream.format("delta").outputMode("append").option("checkpointLocation", BASE + "/checkpoints/windows").trigger(availableNow=True).toTable("window_totals"))
qw.awaitTermination()
display(spark.table("window_totals"))
print(qw.recentProgress)

### Solución 4

In [ ]:
dedup = (spark.readStream.table("events_source").withWatermark("event_time", "10 minutes").dropDuplicatesWithinWatermark(["event_id"]))
qd = (dedup.writeStream.format("delta").option("checkpointLocation", BASE + "/checkpoints/dedup").trigger(availableNow=True).toTable("dedup_events"))
qd.awaitTermination()
display(spark.table("dedup_events"))

### Solución 5

In [ ]:
for progress in qw.recentProgress:
    print({key: progress.get(key) for key in ["batchId", "numInputRows", "durationMs", "eventTime", "stateOperators"]})
# Si la fuente recibe UPDATE/DELETE, usa CDF y aplica los cambios; ver notebook 19.
# skipChangeCommits no convierte actualizaciones en eventos de cambios.

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Qué influye en el watermark?

A. La hora de creación del catálogo

B. El tiempo de evento observado y el umbral

C. El nombre del archivo

D. El salario medio

### Pregunta 2
¿Qué modo escribe todos los grupos de una agregación?

A. append siempre

B. ignore

C. complete

D. overwrite de readStream

### Pregunta 3
¿Es seguro reutilizar el checkpoint para cualquier consulta nueva?

A. No; el estado y el plan deben ser compatibles

B. Sí siempre

C. Solo hay que renombrar el notebook

D. El checkpoint no contiene estado

### Respuestas y explicación
**1. B** — Marca hasta dónde puede cerrarse estado considerando tardanza.

**2. C** — complete emite la tabla de resultados de la agregación.

**3. A** — Cambios de operadores con estado pueden requerir otra consulta y destino.

## PARTE 6 - RETO FINAL
Introduce eventos tardíos y duplicados. Define una tolerancia temporal, ejecuta microlotes separados y justifica qué filas se conservan o pueden descartarse usando las métricas.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
